# kwargs-pass-through-recipe — faded example 1: Pass kwargs to the forward call — fill in the splat

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `kwargs-pass-through-recipe`. Running the beacon reports progress on the `Backprop: Kwargs pass-through` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Kwargs pass-through` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`kwargs-pass-through-recipe`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "kwargs-pass-through-recipe"
DD_SUBTOPIC = "Backprop: Kwargs pass-through"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The wrapped forward function must forward keyword arguments to the underlying numpy/torch function. Writing `fwd_fn(*raw_args)` silently drops all kwargs, causing incorrect reductions. The correct call is `fwd_fn(*raw_args, **kwargs)` so that dimension-specifying arguments like `axis` reach the computation.

## Faded exercise 1

The `tensor_func` body is mostly written. Fill in the forward call so that `kwargs` are forwarded to `fwd_fn` alongside the unboxed positional arguments.

**Fill in:** Call fwd_fn with the unboxed positional arguments and the keyword arguments.

In [ ]:
import numpy as np
from dataclasses import dataclass
from typing import Any, Callable, Optional

@dataclass
class Recipe:
    func: Any
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array):
        self.array = array
        self.recipe: Optional[Recipe] = None

def wrap_forward_fn(fwd_fn: Callable) -> Callable:
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_raw = fwd_fn(*raw_args, **kwargs)
        parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
        out = MiniTensor(out_raw)
        out.recipe = Recipe(fwd_fn, raw_args, dict(kwargs), parents)
        return out
    return tensor_func


def _test():
    import numpy as np

    wrapped_sum = wrap_forward_fn(np.sum)
    x = MiniTensor(np.arange(12.0).reshape(3, 4))

    # With axis kwarg
    out = wrapped_sum(x, axis=1)
    assert out.array.shape == (3,), f'expected (3,) got {out.array.shape}'
    np.testing.assert_allclose(out.array, [6., 22., 38.])
    assert out.recipe.kwargs == {'axis': 1}

    # Without kwargs
    out2 = wrapped_sum(x)
    assert float(out2.array) == 66.0
    assert out2.recipe.kwargs == {}


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import numpy as np
from dataclasses import dataclass
from typing import Any, Callable, Optional

@dataclass
class Recipe:
    func: Any
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array):
        self.array = array
        self.recipe: Optional[Recipe] = None

def wrap_forward_fn(fwd_fn: Callable) -> Callable:
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_raw = fwd_fn(*raw_args, **kwargs)
        parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
        out = MiniTensor(out_raw)
        out.recipe = Recipe(fwd_fn, raw_args, dict(kwargs), parents)
        return out
    return tensor_func
```
</details>